## Data Preparation

This notebook loads raw tourism data from `data/raw/tourism.csv`, applies deterministic cleaning and normalization, and writes cleaned outputs to `data/processed/`.

Outputs:
- `data/processed/cleaned_tourism.csv`: Cleaned tabular dataset suitable for modeling.

Reproducibility notes:
- File paths are relative to the repository root.
- No randomness is used during cleaning.
- Numeric columns are coerced and imputed with median; categorical columns are imputed with mode.
- Minor label fixes are applied (e.g., `Fe Male` -> `Female`).


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import os

# Resolve repo root whether running from Jupyter or headless
this_file = globals().get("__file__")
if this_file is None:
    # When run as a notebook, cwd is typically the notebook dir (e.g., .../notebooks)
    cwd = Path(os.getcwd())
    REPO_ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd
else:
    # When run as a script, jump two levels up from file location
    REPO_ROOT = Path(this_file).resolve().parents[2]

RAW_PATH = REPO_ROOT / "data" / "raw" / "tourism.csv"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RAW_PATH, PROCESSED_DIR


In [ ]:
# Load raw data
assert RAW_PATH.exists(), f"Raw CSV not found at {RAW_PATH}"
df = pd.read_csv(RAW_PATH)

# Drop unnamed index-like columns
for col in list(df.columns):
    if str(col).lower().startswith("unnamed") or col == "":
        df.drop(columns=[col], inplace=True)

# Standardize column names (strip, replace spaces with underscores)
df.columns = [str(c).strip().replace(" ", "_") for c in df.columns]

# Fix obvious label glitches
if "Gender" in df.columns:
    df["Gender"] = df["Gender"].replace({"Fe Male": "Female", "fe male": "Female", "Femail": "Female"})

# Coerce numeric-looking columns
for col in df.columns:
    if df[col].dtype == object:
        # Try numeric coercion where possible without heavy loss of info
        coerced = pd.to_numeric(df[col].str.replace(",", "", regex=False), errors="ignore")
        if not isinstance(coerced, pd.Series) or coerced.dtype == object:
            continue
        # Heuristic: if at least 70% values became numeric, accept coercion
        non_na_ratio = coerced.notna().mean()
        if non_na_ratio >= 0.70:
            df[col] = coerced

df.head()


In [ ]:
# Impute missing values
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in df.columns if c not in numeric_cols]

# Numeric: median
for col in numeric_cols:
    if df[col].isna().any():
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)

# Categorical: mode (most frequent)
for col in cat_cols:
    if df[col].isna().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val.iloc[0])

# Basic sanity checks
assert not df.isna().any().any(), "Unexpected NaNs remain after imputation"
len(df), df.dtypes.head(10)


In [ ]:
# Save cleaned dataset
output_path = PROCESSED_DIR / "cleaned_tourism.csv"
df.to_csv(output_path, index=False)
output_path, output_path.exists(), output_path.stat().st_size if output_path.exists() else None
